In [1]:
import pandas as pd
from pathlib import Path
import re
from IPython.display import display
import numpy as np

In [2]:
#config
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

In [3]:
BASE_DIR = Path.cwd().parent
CHAMPIONS_LEAGUE_DATA_DIR = BASE_DIR / "data" / "raw" / "champions_league"
EUROPA_LEAGUE_DATA_DIR = BASE_DIR / "data" / "raw" / "europa_league"
CONFERENCE_LEAGUE_DATA_DIR = BASE_DIR / "data" / "raw" / "conference_league"

DATA_PROCESSED_DIR = BASE_DIR / "data" / "processed" 

CONFERENCE_LEAGUE_DATA_PROCESSED_DIR = BASE_DIR / "data" / "processed" / "conference_league"
EUROPA_LEAGUE_DATA_PROCESSED_DIR = BASE_DIR / "data" / "processed" / "europa_league"
CHAMPIONS_LEAGUE_DATA_PROCESSED_DIR = BASE_DIR / "data" / "processed" / "champions_league"
df = pd.read_csv(CHAMPIONS_LEAGUE_DATA_DIR / "2021_22_main_stages.csv")

# CHAMPIONS LEAGUE DATA

In [4]:
#checking wheter structure of files are the same across years
champions_league_file_list = list(CHAMPIONS_LEAGUE_DATA_DIR.iterdir())
df_champions_league =[]
for file in champions_league_file_list:
    df_file = pd.read_csv(file)
    print(list(df_file.columns))
    print(f"{file} shape -{df_file.shape}")



['Round', 'Wk', 'Day', 'Date', 'Time', 'Home', 'Score', 'Away', 'Attendance', 'Venue', 'Referee', 'Match Report', 'Notes']
D:\Python Projects\UEFA_coefficient\data\raw\champions_league\2021_22_main_stages.csv shape -(131, 13)
['Round', 'Day', 'Date', 'Time', 'Home', 'Score', 'Away', 'Attendance', 'Venue', 'Referee', 'Match Report', 'Notes']
D:\Python Projects\UEFA_coefficient\data\raw\champions_league\2021_22_qualification.csv shape -(98, 12)
['Round', 'Wk', 'Day', 'Date', 'Time', 'Home', 'Score', 'Away', 'Attendance', 'Venue', 'Referee', 'Match Report', 'Notes']
D:\Python Projects\UEFA_coefficient\data\raw\champions_league\2022_23_main_stages.csv shape -(131, 13)
['Round', 'Day', 'Date', 'Time', 'Home', 'Score', 'Away', 'Attendance', 'Venue', 'Referee', 'Match Report', 'Notes']
D:\Python Projects\UEFA_coefficient\data\raw\champions_league\2022_23_qualification.csv shape -(94, 12)
['Round', 'Wk', 'Day', 'Date', 'Time', 'Home', 'Score', 'Away', 'Attendance', 'Venue', 'Referee', 'Match R

In [5]:
# main stages has more one more column, the rest column have the same naming, so UNIONing is possible
# additional columns are needed for easier sorting / division of the data
dfs = []
for file in champions_league_file_list:
    df_file = pd.read_csv(file)
    filename = str(file).split("\\")[-1]
    competition_phase = ''
    if re.search('main_stages',filename):
        competition_phase = 'main stage'
    elif re.search('qualification',filename):
        competition_phase = 'qualification'
    else:
        'N/A'
        # LOG ERROR HERE AND BREAK CODE !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
        pass

    season = filename.replace("_","/")[:7]
    season_start = filename[:4]
    df_file['Competition phase'] = competition_phase
    df_file['Season'] = season
    df_file['Season start year'] = season_start
    dfs.append(df_file)

df_champions_league = pd.concat(dfs, ignore_index=True)

In [6]:
### data enhancment
# country of club
df_champions_league['Competition'] = 'Champions League'
df_champions_league['Host country'] = df_champions_league['Home'].str.split(' ').str.get(-1)
df_champions_league['Guest country'] = df_champions_league['Away'].str.split(' ').str.get(0)
df_champions_league['Home'] = df_champions_league['Home'].str.rsplit(n=1).str[0]
df_champions_league['Away'] = df_champions_league['Away'].str.split(n=1).str[1]

# penalties addition
df_champions_league['Extra time'] = df_champions_league['Notes'].str.contains('extra time', case=False, regex=False)
df_champions_league['Penalty kicks'] = df_champions_league['Notes'].str.contains('penalty shoot-out', regex=False)
penalties = df_champions_league['Score'].str.findall(r'\((\d+)\)')
df_champions_league['Host penalties scored'] = penalties.apply(lambda x: x[0] if isinstance(x, list) and len(x) >= 2 else None)
df_champions_league['Guest penalties scored'] = penalties.apply(lambda x: x[-1] if isinstance(x, list) and len(x) >= 2 else None)
df_champions_league['Penalty kicks result'] = df_champions_league['Host penalties scored'].astype('string')+ ' - '+ df_champions_league['Guest penalties scored'].astype('string')

# result after base time + extra time
result_pattern = r'(?:\(\d+\)\s*)?(\d+)\s*[–-]\s*(\d+)(?:\s*\(\d+\))?'
result = df_champions_league['Score'].str.extract(result_pattern)
df_champions_league['Host goals'] = pd.to_numeric(result[0]).astype('Int64')
df_champions_league['Guest goals'] = pd.to_numeric(result[1]).astype('Int64')

In [7]:
### data cleaning
# getting rid of empty rows
df_champions_league = df_champions_league.dropna(subset=['Home'])

# clearing data from columns
df_champions_league['Time'] = df_champions_league['Time'].apply(lambda x: str(x).split(' ')[0])

# assignment of proper data types 
df_champions_league['Wk'] = df_champions_league['Wk'].astype('Int64')
df_champions_league['Attendance'] = df_champions_league['Attendance'].astype('Int64')
df_champions_league['Host country'] = df_champions_league['Host country'].astype('str')
df_champions_league['Home'] = df_champions_league['Home'].astype('str')
df_champions_league['Guest country'] = df_champions_league['Guest country'].astype('str')
df_champions_league['Away'] = df_champions_league['Away'].astype('str')
df_champions_league['Date'] = pd.to_datetime(df_champions_league['Date'], format="%Y-%m-%d")
df_champions_league['Penalty kicks result'] = df_champions_league['Penalty kicks result'].astype('str')
df_champions_league['Host penalties scored'] = df_champions_league['Host penalties scored'].astype('Int64')
df_champions_league['Guest penalties scored'] = df_champions_league['Guest penalties scored'].astype('Int64')
df_champions_league['Extra time'] = df_champions_league['Extra time'].astype('boolean')
df_champions_league['Penalty kicks'] = df_champions_league['Penalty kicks'].astype('boolean')
df_champions_league['Notes'] = df_champions_league['Notes'].astype('str')
df_champions_league['Season start year'] = df_champions_league['Season start year'].astype('Int64')

	
# columns renaming
df_champions_league.rename(columns=
                           {
                           'Wk': 'Matchweek',
                           'Home': 'Host',
                           'Away': 'Guest'
                           }, 
                           inplace=True
                          )
# changing columns order
df_champions_league = df_champions_league[
    [
     'Competition',
     'Season',
     'Season start year',
     'Round',
     'Matchweek',
     'Day',
     'Date',
     'Time',
     'Host country',
     'Host',
     'Host goals',
     'Score',
     'Guest country',
     'Guest',
     'Guest goals',
     'Attendance',
     'Venue',
     'Referee',
     'Notes',
     'Competition phase',
     'Extra time',
     'Penalty kicks',
     'Penalty kicks result',
     'Host penalties scored',
     'Guest penalties scored']
     
]


In [8]:
df_champions_league

,Competition,Season,Season start year,Round,Matchweek,Day,Date,Time,Host country,Host,Host goals,Score,Guest country,Guest,Guest goals,Attendance,Venue,Referee,Notes,Competition phase,Extra time,Penalty kicks,Penalty kicks result,Host penalties scored,Guest penalties scored
0,Champions League,2021/22,2021,Group stage,1,Tue,2021-09-14,18:45,es,Sevilla,1,1–1,at,RB Salzburg,1,18373,Estadio Ramón Sánchez Pizjuán,Aliaksei Kulbakou,NaN,main stage,<NA>,<NA>,NaN,<NA>,<NA>
1,Champions League,2021/22,2021,Group stage,1,Tue,2021-09-14,18:45,ch,Young Boys,2,2–1,eng,Manchester Utd,1,31120,Wankdorfstadion,François Letexier,NaN,main stage,<NA>,<NA>,NaN,<NA>,<NA>
2,Champions League,2021/22,2021,Group stage,1,Tue,2021-09-14,20:00,eng,Chelsea,1,1–0,ru,Zenit,0,39252,Stamford Bridge,Bartosz Frankowski,NaN,main stage,<NA>,<NA>,NaN,<NA>,<NA>
3,Champions League,2021/22,2021,Group stage,1,Tue,2021-09-14,21:00,es,Villarreal,2,2–2,it,Atalanta,2,12916,Estadio de la Cerámica,Clément Turpin,NaN,main stage,<NA>,<NA>,NaN,<NA>,<NA>
4,Champions League,2021/22,2021,Group stage,1,Tue,2021-09-14,21:00,es,Barcelona,0,0–3,de,Bayern Munich,3,39737,Camp Nou,Michael Oliver,NaN,main stage,<NA>,<NA>,NaN,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1500,Champions League,2026/27,2026,Play-off round,<NA>,Tue,2026-08-25,21:00,no,Bodø/Glimt,3,3–0,nl,NEC Nijmegen,0,8009,Aspmyra Stadion,Michael Oliver,Leg 2 of 2; Bodø/Glimt won,qualification,False,False,NaN,<NA>,<NA>
1501,Champions League,2026/27,2026,Play-off round,<NA>,Wed,2026-08-26,21:00,fr,Lyon,1,1–2,tr,Fenerbahçe,2,55443,Groupama Stadium,Maurizio Mariani,Leg 2 of 2; Fenerbahçe won,qualification,False,False,NaN,<NA>,<NA>
1502,Champions League,2026/27,2026,Play-off round,<NA>,Wed,2026-08-26,21:00,no,Viking,3,3–1,hr,Dinamo Zagreb,1,15152,Lyse Arena,Jose Maria Sánchez,Leg 2 of 2; Viking won,qualification,False,False,NaN,<NA>,<NA>
1503,Champions League,2026/27,2026,Play-off round,<NA>,Wed,2026-08-26,21:00,si,NK Celje,1,1–2,sk,Slovan Bratislava,2,11500,Stadion Z'dežele,Alejandro Hernández,Leg 2 of 2; Slovan Bratislava won; Required Ex...,qualification,True,False,NaN,<NA>,<NA>


In [9]:
df_champions_league
df_champions_league.to_csv(f"{DATA_PROCESSED_DIR}/champions_league_matches.csv",index=False)


In [10]:
df_champions_league.dtypes

Competition                          str
Season                               str
Season start year                  Int64
Round                                str
Matchweek                          Int64
Day                                  str
Date                      datetime64[us]
Time                                 str
Host country                         str
Host                                 str
Host goals                         Int64
Score                                str
Guest country                        str
Guest                                str
Guest goals                        Int64
Attendance                         Int64
Venue                                str
Referee                              str
Notes                                str
Competition phase                    str
Extra time                       boolean
Penalty kicks                    boolean
Penalty kicks result                 str
Host penalties scored              Int64
Guest penalties 

# EUROPA LEAGUE

In [11]:
#checking wheter structure of files are the same across years
europa_league_file_list = list(EUROPA_LEAGUE_DATA_DIR.iterdir())
df_europa_league =[]
for file in europa_league_file_list:
    df_file = pd.read_csv(file)
    print(list(df_file.columns))
    print(f"{file} shape -{df_file.shape}")


['Round', 'Wk', 'Day', 'Date', 'Time', 'Home', 'Score', 'Away', 'Attendance', 'Venue', 'Referee', 'Match Report', 'Notes']
D:\Python Projects\UEFA_coefficient\data\raw\europa_league\2021_22_main_stages.csv shape -(147, 13)
['Round', 'Day', 'Date', 'Time', 'Home', 'Score', 'Away', 'Attendance', 'Venue', 'Referee', 'Match Report', 'Notes']
D:\Python Projects\UEFA_coefficient\data\raw\europa_league\2021_22_qualification.csv shape -(37, 12)
['Round', 'Wk', 'Day', 'Date', 'Time', 'Home', 'Score', 'Away', 'Attendance', 'Venue', 'Referee', 'Match Report', 'Notes']
D:\Python Projects\UEFA_coefficient\data\raw\europa_league\2022_23_main_stages.csv shape -(148, 13)
['Round', 'Day', 'Date', 'Time', 'Home', 'Score', 'Away', 'Attendance', 'Venue', 'Referee', 'Match Report', 'Notes']
D:\Python Projects\UEFA_coefficient\data\raw\europa_league\2022_23_qualification.csv shape -(35, 12)
['Round', 'Wk', 'Day', 'Date', 'Time', 'Home', 'Score', 'Away', 'Attendance', 'Venue', 'Referee', 'Match Report', 'Not

In [12]:
# main stages has more one more column, the rest column have the same naming, so UNIONing is possible
# additional columns are needed for easier sorting / division of the data
dfs = []
for file in europa_league_file_list:
    df_file = pd.read_csv(file)
    filename = str(file).split("\\")[-1]
    competition_phase = ''
    if re.search('main_stages',filename):
        competition_phase = 'main stage'
    elif re.search('qualification',filename):
        competition_phase = 'qualification'
    else:
        'N/A'
        # LOG ERROR HERE AND BREAK CODE !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
        pass

    season = filename.replace("_","/")[:7]
    season_start = filename[:4]
    df_file['Competition phase'] = competition_phase
    df_file['Season'] = season
    df_file['Season start year'] = season_start
    dfs.append(df_file)

df_europa_league = pd.concat(dfs, ignore_index=True)

In [13]:
### data enhancment
# country of club
df_europa_league['Competition'] = 'Europa League'
df_europa_league['Host country'] = df_europa_league['Home'].str.split(' ').str.get(-1)
df_europa_league['Guest country'] = df_europa_league['Away'].str.split(' ').str.get(0)
df_europa_league['Home'] = df_europa_league['Home'].str.rsplit(n=1).str[0]
df_europa_league['Away'] = df_europa_league['Away'].str.split(n=1).str[1]

# penalties addition
df_europa_league['Extra time'] = df_europa_league['Notes'].str.contains('extra time', case=False, regex=False)
df_europa_league['Penalty kicks'] = df_europa_league['Notes'].str.contains('penalty shoot-out', regex=False)
penalties = df_europa_league['Score'].str.findall(r'\((\d+)\)')
df_europa_league['Host penalties scored'] = penalties.apply(lambda x: x[0] if isinstance(x, list) and len(x) >= 2 else None)
df_europa_league['Guest penalties scored'] = penalties.apply(lambda x: x[-1] if isinstance(x, list) and len(x) >= 2 else None)
df_europa_league['Penalty kicks result'] = df_europa_league['Host penalties scored'].astype('string')+ ' - '+ df_europa_league['Guest penalties scored'].astype('string')

# result after base time + extra time
result_pattern = r'(?:\(\d+\)\s*)?(\d+)\s*[–-]\s*(\d+)(?:\s*\(\d+\))?'
result = df_europa_league['Score'].str.extract(result_pattern)
df_europa_league['Host goals'] = pd.to_numeric(result[0]).astype('Int64')
df_europa_league['Guest goals'] = pd.to_numeric(result[1]).astype('Int64')

In [14]:
### data cleaning
# getting rid of empty rows
df_europa_league = df_europa_league.dropna(subset=['Home'])

# clearing data from columns
df_europa_league['Time'] = df_europa_league['Time'].apply(lambda x: str(x).split(' ')[0])

# assignment of proper data types 
df_europa_league['Wk'] = df_europa_league['Wk'].astype('Int64')
df_europa_league['Attendance'] = df_europa_league['Attendance'].astype('Int64')
df_europa_league['Host country'] = df_europa_league['Host country'].astype('str')
df_europa_league['Home'] = df_europa_league['Home'].astype('str')
df_europa_league['Guest country'] = df_europa_league['Guest country'].astype('str')
df_europa_league['Away'] = df_europa_league['Away'].astype('str')
df_europa_league['Date'] = pd.to_datetime(df_europa_league['Date'], format="%Y-%m-%d")
df_europa_league['Penalty kicks result'] = df_europa_league['Penalty kicks result'].astype('str')
df_europa_league['Host penalties scored'] = df_europa_league['Host penalties scored'].astype('Int64')
df_europa_league['Guest penalties scored'] = df_europa_league['Guest penalties scored'].astype('Int64')
df_europa_league['Extra time'] = df_europa_league['Extra time'].astype('boolean')
df_europa_league['Penalty kicks'] = df_europa_league['Penalty kicks'].astype('boolean')
df_europa_league['Notes'] = df_europa_league['Notes'].astype('str')
df_europa_league['Season start year'] = df_europa_league['Season start year'].astype('Int64')

# columns renaming
df_europa_league.rename(columns=
                           {
                           'Wk': 'Matchweek',
                           'Home': 'Host',
                           'Away': 'Guest'
                           }, 
                           inplace=True
                          )
# changing columns order
df_europa_league = df_europa_league[
    [
     'Competition',
     'Season',
     'Season start year',
     'Round',
     'Matchweek',
     'Day',
     'Date',
     'Time',
     'Host country',
     'Host',
     'Host goals',
     'Score',
     'Guest country',
     'Guest',
     'Guest goals',
     'Attendance',
     'Venue',
     'Referee',
     'Notes',
     'Competition phase',
     'Extra time',
     'Penalty kicks',
     'Penalty kicks result',
     'Host penalties scored',
     'Guest penalties scored']
     
]


In [15]:
df_europa_league
df_europa_league.to_csv(f"{DATA_PROCESSED_DIR}/europa_league_matches.csv",index=False)

In [16]:
df_europa_league

,Competition,Season,Season start year,Round,Matchweek,Day,Date,Time,Host country,Host,Host goals,Score,Guest country,Guest,Guest goals,Attendance,Venue,Referee,Notes,Competition phase,Extra time,Penalty kicks,Penalty kicks result,Host penalties scored,Guest penalties scored
0,Europa League,2021/22,2021,Group stage,1,Wed,2021-09-15,17:30,ru,Spartak Moscow,0,0–1,pl,Legia Warsaw,1,6832,Stadion Akademii imeni Cherenkova № 4,Nikola Dabanović,NaN,main stage,<NA>,<NA>,NaN,<NA>,<NA>
1,Europa League,2021/22,2021,Group stage,1,Thu,2021-09-16,18:45,es,Real Betis,4,4–3,sct,Celtic,3,30893,Estadio Benito Villamarín,Fran Jović,NaN,main stage,<NA>,<NA>,NaN,<NA>,<NA>
2,Europa League,2021/22,2021,Group stage,1,Thu,2021-09-16,18:45,at,Rapid Wien,0,0–1,be,Genk,1,18400,Gerhard-Hanappi-Stadion,Kristo Tohver,NaN,main stage,<NA>,<NA>,NaN,<NA>,<NA>
3,Europa League,2021/22,2021,Group stage,1,Thu,2021-09-16,18:45,rs,Red Star,2,2–1,pt,Braga,1,24671,Stadion Rajko Mitić,John Beaton,NaN,main stage,<NA>,<NA>,NaN,<NA>,<NA>
4,Europa League,2021/22,2021,Group stage,1,Thu,2021-09-16,18:45,de,Leverkusen,2,2–1,hu,Ferencváros,1,11013,BayArena,Andris Treimanis,NaN,main stage,<NA>,<NA>,NaN,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1341,Europa League,2026/27,2026,Play-off round,<NA>,Thu,2026-08-27,20:00,ch,Thun,2,2–2,pl,Lech Poznań,2,5223,Stockhorn Arena,Halil Umut Meler,Leg 2 of 2; Lech Poznań won,qualification,False,False,NaN,<NA>,<NA>
1342,Europa League,2026/27,2026,Play-off round,<NA>,Thu,2026-08-27,20:00,dk,AGF,1,1–3,pt,Benfica,3,8945,Cepheus Park Randers,Daniel Siebert,Leg 2 of 2; Benfica won,qualification,False,False,NaN,<NA>,<NA>
1343,Europa League,2026/27,2026,Play-off round,<NA>,Thu,2026-08-27,20:30,hu,Ferencváros,4,4–0,tr,Trabzonspor,0,21011,Groupama Arena,Rade Obrenović,Leg 2 of 2; Ferencváros won,qualification,False,False,NaN,<NA>,<NA>
1344,Europa League,2026/27,2026,Play-off round,<NA>,Thu,2026-08-27,20:30,be,Anderlecht,3,3–0,kz,FC Kairat,0,16861,Lotto Park,Espen Eskås,Leg 2 of 2; Anderlecht won,qualification,False,False,NaN,<NA>,<NA>


# CONFERENCE LEAGUE

In [17]:
#checking wheter structure of files are the same across years
conference_league_file_list = list(CONFERENCE_LEAGUE_DATA_DIR.iterdir())
df_conference_league =[]
for file in conference_league_file_list:
    df_file = pd.read_csv(file)
    print(list(df_file.columns))
    print(f"{file} shape -{df_file.shape}")


['Round', 'Wk', 'Day', 'Date', 'Time', 'Home', 'Score', 'Away', 'Attendance', 'Venue', 'Referee', 'Match Report', 'Notes']
D:\Python Projects\UEFA_coefficient\data\raw\conference_league\2021_22_main_stages.csv shape -(147, 13)
['Round', 'Day', 'Date', 'Time', 'Home', 'Score', 'Away', 'Attendance', 'Venue', 'Referee', 'Match Report', 'Notes']
D:\Python Projects\UEFA_coefficient\data\raw\conference_league\2021_22_qualification.csv shape -(285, 12)
['Round', 'Wk', 'Day', 'Date', 'Time', 'Home', 'Score', 'Away', 'Attendance', 'Venue', 'Referee', 'Match Report', 'Notes']
D:\Python Projects\UEFA_coefficient\data\raw\conference_league\2022_23_main_stages.csv shape -(147, 13)
['Round', 'Day', 'Date', 'Time', 'Home', 'Score', 'Away', 'Attendance', 'Venue', 'Referee', 'Match Report', 'Notes']
D:\Python Projects\UEFA_coefficient\data\raw\conference_league\2022_23_qualification.csv shape -(277, 12)
['Round', 'Wk', 'Day', 'Date', 'Time', 'Home', 'Score', 'Away', 'Attendance', 'Venue', 'Referee', 'M

In [18]:
# main stages has more one more column, the rest column have the same naming, so UNIONing is possible
# additional columns are needed for easier sorting / division of the data
dfs = []
for file in europa_league_file_list:
    df_file = pd.read_csv(file)
    filename = str(file).split("\\")[-1]
    competition_phase = ''
    if re.search('main_stages',filename):
        competition_phase = 'main stage'
    elif re.search('qualification',filename):
        competition_phase = 'qualification'
    else:
        'N/A'
        # LOG ERROR HERE AND BREAK CODE !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
        pass

    season = filename.replace("_","/")[:7]
    season_start = filename[:4]
    df_file['Competition phase'] = competition_phase
    df_file['Season'] = season
    df_file['Season start year'] = season_start
    dfs.append(df_file)

df_conference_league = pd.concat(dfs, ignore_index=True)

In [19]:
### data enhancment
# country of club
df_conference_league['Competition'] = 'Conference League'
df_conference_league['Host country'] = df_conference_league['Home'].str.split(' ').str.get(-1)
df_conference_league['Guest country'] = df_conference_league['Away'].str.split(' ').str.get(0)
df_conference_league['Home'] = df_conference_league['Home'].str.rsplit(n=1).str[0]
df_conference_league['Away'] = df_conference_league['Away'].str.split(n=1).str[1]

# penalties addition
df_conference_league['Extra time'] = df_conference_league['Notes'].str.contains('extra time', case=False, regex=False)
df_conference_league['Penalty kicks'] = df_conference_league['Notes'].str.contains('penalty shoot-out', regex=False)
penalties = df_conference_league['Score'].str.findall(r'\((\d+)\)')
df_conference_league['Host penalties scored'] = penalties.apply(lambda x: x[0] if isinstance(x, list) and len(x) >= 2 else None)
df_conference_league['Guest penalties scored'] = penalties.apply(lambda x: x[-1] if isinstance(x, list) and len(x) >= 2 else None)
df_conference_league['Penalty kicks result'] = df_conference_league['Host penalties scored'].astype('string')+ ' - '+ df_conference_league['Guest penalties scored'].astype('string')

# result after base time + extra time
result_pattern = r'(?:\(\d+\)\s*)?(\d+)\s*[–-]\s*(\d+)(?:\s*\(\d+\))?'
result = df_conference_league['Score'].str.extract(result_pattern)
df_conference_league['Host goals'] = pd.to_numeric(result[0]).astype('Int64')
df_conference_league['Guest goals'] = pd.to_numeric(result[1]).astype('Int64')

In [20]:
### data cleaning
# getting rid of empty rows
df_conference_league = df_conference_league.dropna(subset=['Home'])

# clearing data from columns
df_conference_league['Time'] = df_conference_league['Time'].apply(lambda x: str(x).split(' ')[0])

# assignment of proper data types 
df_conference_league['Wk'] = df_conference_league['Wk'].astype('Int64')
df_conference_league['Attendance'] = df_conference_league['Attendance'].astype('Int64')
df_conference_league['Host country'] = df_conference_league['Host country'].astype('str')
df_conference_league['Home'] = df_conference_league['Home'].astype('str')
df_conference_league['Guest country'] = df_conference_league['Guest country'].astype('str')
df_conference_league['Away'] = df_conference_league['Away'].astype('str')
df_conference_league['Date'] = pd.to_datetime(df_conference_league['Date'], format="%Y-%m-%d")
df_conference_league['Penalty kicks result'] = df_conference_league['Penalty kicks result'].astype('str')
df_conference_league['Host penalties scored'] = df_conference_league['Host penalties scored'].astype('Int64')
df_conference_league['Guest penalties scored'] = df_conference_league['Guest penalties scored'].astype('Int64')
df_conference_league['Extra time'] = df_conference_league['Extra time'].astype('boolean')
df_conference_league['Penalty kicks'] = df_conference_league['Penalty kicks'].astype('boolean')
df_conference_league['Notes'] = df_conference_league['Notes'].astype('str')
df_conference_league['Season start year'] = df_conference_league['Season start year'].astype('Int64')

# columns renaming
df_conference_league.rename(columns=
                           {
                           'Wk': 'Matchweek',
                           'Home': 'Host',
                           'Away': 'Guest'
                           }, 
                           inplace=True
                          )
# changing columns order
df_conference_league = df_conference_league[
    [
     'Competition',
     'Season',
     'Season start year',
     'Round',
     'Matchweek',
     'Day',
     'Date',
     'Time',
     'Host country',
     'Host',
     'Host goals',
     'Score',
     'Guest country',
     'Guest',
     'Guest goals',
     'Attendance',
     'Venue',
     'Referee',
     'Notes',
     'Competition phase',
     'Extra time',
     'Penalty kicks',
     'Penalty kicks result',
     'Host penalties scored',
     'Guest penalties scored']
     
]


In [21]:
df_conference_league
df_conference_league.to_csv(f"{DATA_PROCESSED_DIR}/conference_league_matches.csv",index=False)


In [22]:
df_european_competition = pd.concat([df_conference_league,df_europa_league,df_champions_league], ignore_index=True)
df_european_competition.to_csv(f"{DATA_PROCESSED_DIR}/all_european_competition_matches.csv",index=False)


In [23]:
df_european_competition

,Competition,Season,Season start year,Round,Matchweek,Day,Date,Time,Host country,Host,Host goals,Score,Guest country,Guest,Guest goals,Attendance,Venue,Referee,Notes,Competition phase,Extra time,Penalty kicks,Penalty kicks result,Host penalties scored,Guest penalties scored
0,Conference League,2021/22,2021,Group stage,1,Wed,2021-09-15,17:30,ru,Spartak Moscow,0,0–1,pl,Legia Warsaw,1,6832,Stadion Akademii imeni Cherenkova № 4,Nikola Dabanović,NaN,main stage,<NA>,<NA>,NaN,<NA>,<NA>
1,Conference League,2021/22,2021,Group stage,1,Thu,2021-09-16,18:45,es,Real Betis,4,4–3,sct,Celtic,3,30893,Estadio Benito Villamarín,Fran Jović,NaN,main stage,<NA>,<NA>,NaN,<NA>,<NA>
2,Conference League,2021/22,2021,Group stage,1,Thu,2021-09-16,18:45,at,Rapid Wien,0,0–1,be,Genk,1,18400,Gerhard-Hanappi-Stadion,Kristo Tohver,NaN,main stage,<NA>,<NA>,NaN,<NA>,<NA>
3,Conference League,2021/22,2021,Group stage,1,Thu,2021-09-16,18:45,rs,Red Star,2,2–1,pt,Braga,1,24671,Stadion Rajko Mitić,John Beaton,NaN,main stage,<NA>,<NA>,NaN,<NA>,<NA>
4,Conference League,2021/22,2021,Group stage,1,Thu,2021-09-16,18:45,de,Leverkusen,2,2–1,hu,Ferencváros,1,11013,BayArena,Andris Treimanis,NaN,main stage,<NA>,<NA>,NaN,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4017,Champions League,2026/27,2026,Play-off round,<NA>,Tue,2026-08-25,21:00,no,Bodø/Glimt,3,3–0,nl,NEC Nijmegen,0,8009,Aspmyra Stadion,Michael Oliver,Leg 2 of 2; Bodø/Glimt won,qualification,False,False,NaN,<NA>,<NA>
4018,Champions League,2026/27,2026,Play-off round,<NA>,Wed,2026-08-26,21:00,fr,Lyon,1,1–2,tr,Fenerbahçe,2,55443,Groupama Stadium,Maurizio Mariani,Leg 2 of 2; Fenerbahçe won,qualification,False,False,NaN,<NA>,<NA>
4019,Champions League,2026/27,2026,Play-off round,<NA>,Wed,2026-08-26,21:00,no,Viking,3,3–1,hr,Dinamo Zagreb,1,15152,Lyse Arena,Jose Maria Sánchez,Leg 2 of 2; Viking won,qualification,False,False,NaN,<NA>,<NA>
4020,Champions League,2026/27,2026,Play-off round,<NA>,Wed,2026-08-26,21:00,si,NK Celje,1,1–2,sk,Slovan Bratislava,2,11500,Stadion Z'dežele,Alejandro Hernández,Leg 2 of 2; Slovan Bratislava won; Required Ex...,qualification,True,False,NaN,<NA>,<NA>


In [24]:

df_european_competition.dtypes

Competition                          str
Season                               str
Season start year                  Int64
Round                                str
Matchweek                          Int64
Day                                  str
Date                      datetime64[us]
Time                                 str
Host country                         str
Host                                 str
Host goals                         Int64
Score                                str
Guest country                        str
Guest                                str
Guest goals                        Int64
Attendance                         Int64
Venue                                str
Referee                              str
Notes                                str
Competition phase                    str
Extra time                       boolean
Penalty kicks                    boolean
Penalty kicks result                 str
Host penalties scored              Int64
Guest penalties 